In [ ]:
import os
from pathlib import Path

import autoroot
import hydra
import matplotlib.pyplot as plt
import torch

from src.functionals.vandermonde import *
from src.modules.layers.diagRNN import diagRNN, diagRNN2
from src.modules.lit_module import LitModule
from src.utils import initialize_resolvers

initialize_resolvers()

In [ ]:
hidden_dim = 10
batch_size = 2
seq_length = 10
input_dim = 1

model = diagRNN(input_dim, 1, hidden_dim)
c = model.c
W = model.W
U = model.U

model2 = diagRNN2(input_dim, 1, hidden_dim)
c2 = model2.c
W1 = model2.W1
W2 = model2.W2
U1 = model2.U1
U2 = model2.U2

In [ ]:
inputs = torch.rand(batch_size, seq_length, input_dim)
yhat1 = model(inputs)
yhat2 = model2(inputs)

In [ ]:
outputs1 = []
for t in range(seq_length):
    y = 0
    for s in range(t + 1):
        y += inputs[:, t - s, :] @ U.T @ torch.diag(W**s) @ c.T
    outputs1.append(y)
outputs1 = torch.cat(outputs1, dim=-1).unsqueeze(dim=-1)

In [ ]:
torch.allclose(yhat1, outputs1)

In [ ]:
outputs2 = []
for t in range(seq_length):
    y = 0
    for s in range(t + 1):
        rho = 0
        for r in range(s + 1):
            rho += U1.T @ torch.diag(W1 ** (s - r)) @ U2.T @ torch.diag(W2 ** (r)) @ c2.T
        y += inputs[:, t - s, :] @ rho
    outputs2.append(y)
outputs2 = torch.cat(outputs2, dim=-1).unsqueeze(dim=-1)

In [ ]:
model = LitModule.load_from_checkpoint(
    "./logs/train/2024-05-27_15-03-36/checkpoints/epoch_001-6.597e-01.ckpt"
)
c = model.model.layers[0].c
W = model.model.layers[0].W
U = model.model.layers[0].U

In [ ]:
import numpy as np

plt.plot([(U.T @ torch.diag(W**s) @ c.T).item() for s in range(100)])

In [ ]:
torch.linalg.eigvals(W)